> ## SOLUTION / ANSWER KEY &mdash; Lab 6.2
> This is the **completed** notebook (all `___` blanks filled). For the student version, open
> [`../lab-02-tool-descriptions.ipynb`](../lab-02-tool-descriptions.ipynb). Trainer use &mdash; or self-check after you've tried it yourself.

# Lab 6.2 &mdash; The Docstring Is the Tool's API

**Level:** Beginner &nbsp;|&nbsp; **Est. time:** 20 min &nbsp;|&nbsp; **Day 3 &middot; Module 6 &mdash; Frameworks for Building AI Agents**

### What you'll do
- Write descriptions that tell the model WHEN to use each tool
- Render the catalog (name + description) the model actually reads
- Route a request to the right tool from that catalog

> **How this lab works (experiential flow):** read the **Concept**, run the **Demo** to see it work, then complete **Your Turn** by replacing every `___` placeholder. Run the **grader** cell at the end &mdash; it prints `[PASS]` / `[FAIL]` / `[TODO]` and a final `Score`. Aim for a full score.

> **Framework note:** the graded steps use a tiny **LangChain-shaped shim** (same names &amp; shapes as real LangChain) so they run offline &amp; deterministically. Advanced labs end with an **optional** cell that runs the **real** library.

**Reference:** [Module 6 slides &mdash; Defining a tool](../../presentation/day3-module6-frameworks-for-building-ai-agents.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 6 labs](./index.html)

In [ ]:
# Setup -- run me first
import os
WORK = "/tmp/biaa-lab-06-02"
os.makedirs(WORK, exist_ok=True)
print("Working dir:", WORK)

## Concept
The model never sees your code &mdash; only each tool's **name**, **description** and **args**. So the
**docstring is the tool's real API**: a vague one makes the agent pick the wrong tool. Here you write
good descriptions, build the **catalog** the model is shown, and route a query to a tool from it.

In [ ]:
# --- LangChain-SHAPED shim: a tool has .name, .description (from the docstring), .args, .invoke() ---
import inspect
class Tool:
    def __init__(self, fn, name=None, description=None):
        self.fn = fn
        self.name = name or fn.__name__
        self.description = (description or inspect.getdoc(fn) or "").strip()
        self.args = list(inspect.signature(fn).parameters)
    def invoke(self, value):
        return self.fn(**value) if isinstance(value, dict) else self.fn(value)
    def __repr__(self):
        return "Tool(name=%r)" % self.name
def tool(fn):
    """The @tool decorator -- wrap a plain function into a Tool (mirrors langchain_core.tools.tool)."""
    return Tool(fn)

@tool
def clock(_):
    """Return the current time of day."""
    return "12:00"
print("the model is shown -> ", clock.name + ": " + clock.description)

## Your Turn
Write three tools with clear descriptions, render the catalog, then complete `choose_tool`.

In [ ]:
@tool
def weather(city):
    """Get the current weather for a city. Use for questions about current temperature or conditions."""
    return "sunny, 24C"

@tool
def calculator(expression):
    """Do exact arithmetic on a math expression. Use for any calculation with numbers."""
    return "(computed)"

@tool
def translate(text):
    """Translate the given text into French."""
    return "(translated)"

TOOLS = [weather, calculator, translate]

def tool_catalog(tools):
    # the text the MODEL is shown to choose a tool
    return "\n".join(t.name + ": " + t.description for t in tools)

def choose_tool(query):
    q = query.lower()
    if "weather" in q or "temperature" in q:
        return "weather"
    if "translate" in q:
        return "translate"
    if any(ch.isdigit() for ch in q):
        return "calculator"
    return "weather"

try:
    print(tool_catalog(TOOLS))
    print('---')
    for q in ['weather in tokyo', 'what is 12*8', 'translate hi to french']:
        print(q, '->', choose_tool(q))
except Exception as e:
    print('Fill the blanks, then re-run.', type(e).__name__)

In [ ]:
# === Auto-grader: run after filling the blanks above ===
_results = []
def _rec(label, status, extra=""):
    _results.append(status); print(f"[{status}] {label}" + (f" -- {extra}" if extra else ""))
def expect(label, got, want):
    if got == "___" or got is None: _rec(label, "TODO")
    elif got == want: _rec(label, "PASS")
    else: _rec(label, "FAIL", f"got {got!r}")
def expect_true(label, fn):
    try: _rec(label, "PASS" if fn() else "FAIL")
    except Exception as e: _rec(label, "TODO", type(e).__name__)

expect_true("catalog lists all three tool names", lambda: all(n in tool_catalog(TOOLS) for n in ("weather", "calculator", "translate")))
expect_true("catalog has one line per tool", lambda: tool_catalog(TOOLS).count(chr(10)) == 2)
expect_true("a weather query routes to weather", lambda: choose_tool("weather in paris") == "weather")
expect_true("a numeric query routes to calculator", lambda: choose_tool("what is 12*8") == "calculator")
expect_true("a translate query routes to translate", lambda: choose_tool("translate hello to french") == "translate")

_p = _results.count("PASS")
print(f"\nScore: {_p}/{len(_results)}")
print("All checks passed -- lab complete!" if _p == len(_results) else "Keep going: fill the blanks marked ___ and re-run.")

---
### Key takeaway
The description is what the model reads to choose a tool -- write it for the model. Real LangChain selects from exactly this catalog; here you made it visible.

[&#8592; All Module 6 labs](./index.html) &nbsp;&middot;&nbsp; [Module 6 slides](../../presentation/day3-module6-frameworks-for-building-ai-agents.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>